# SQL query from table names - Continued

In [1]:
from openai import OpenAI
import os

from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

## The old Prompt

In [2]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [3]:
context = [ {'role':'system', 'content':"""
 CREATE SEVERAL (3+) TABLES HERE
"""} ]



In [4]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
WRITE IN YOUR CONTEXT QUERIES HERE
"""
})

In [5]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [7]:
#new
context_user = context.copy()
print(return_CCRMSQL("""What is the average salary depending on the age?""", context_user))

```sql
SELECT age, AVG(salary) AS average_salary
FROM employees
GROUP BY age;
```


In [9]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("What is the average salary depending on the age?", old_context_user))

This is your SQL:
```sql
SELECT AVG(salary) AS average_salary
FROM salary
GROUP BY year
```

This SQL query calculates the average salary for each year by grouping the salaries based on the year they were recorded.


In [10]:
#new
print(return_CCRMSQL("What is the average salary depending on the age?", context_user))

```sql
SELECT age, AVG(salary) AS average_salary
FROM employees
GROUP BY age;
```


In [11]:
#old
print(return_CCRMSQL("What is the average salary depending on the age?", old_context_user))

This is your SQL:
```sql
SELECT AVG(salary) AS average_salary
FROM salary
GROUP BY year;
```
This SQL query calculates the average salary for each year by grouping the salaries based on the year they were received.


# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [12]:
# ---------------------------------------------
# Improved Prompt for SQL Assistant
# ---------------------------------------------
from openai import OpenAI

# Replace with your actual API key
# OPENAI_API_KEY = "your_api_key_here"

# --- SQL Assistant System Prompt ---
context = [
    {'role': 'system', 'content': """
You are SQLHelper, an expert SQL assistant.
Your job is to generate simple, efficient SQL queries based on user questions.

Rules:
- All your answers must start with: "This is your SQL:"
- Then show the SQL command formatted in a code block with syntax highlighting.
- After the SQL, add a short (1–2 sentences) explanation of what the query does.
- Keep SQL syntax valid and simple (SQLite syntax preferred).
- If the question cannot be solved with SQL, respond politely with fewer than 10 words,
  suggesting the user ask something that can be answered using SQL.

Database Schema:
Below are the available tables and their structures.
You must use them when generating SQL queries.
"""}
]

# --- Define tables (improved JSON-like structure for clarity) ---
table_definitions = [
    {
        "table_name": "employees",
        "fields": {
            "ID_usr": "INT",
            "name": "VARCHAR"
        }
    },
    {
        "table_name": "salary",
        "fields": {
            "ID_usr": "INT",
            "year": "DATE",
            "salary": "FLOAT"
        }
    },
    {
        "table_name": "studies",
        "fields": {
            "ID": "INT",
            "ID_usr": "INT",
            "educational_level": "INT",
            "institution": "VARCHAR",
            "years": "DATE",
            "speciality": "VARCHAR"
        }
    }
]

# Add the table definitions into the system context
for table in table_definitions:
    context.append({'role': 'system', 'content': f"Table: {table['table_name']} | Fields: {table['fields']}"})

# --- Function to query the model ---
def return_CCRMSQL(user_message, context):
    client = OpenAI(api_key=OPENAI_API_KEY)

    messages = context.copy()
    messages.append({'role': 'user', 'content': f"Question: {user_message}"})

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        temperature=0
    )

    return response.choices[0].message.content


# ---------------------------------------------
# Example Question
# ---------------------------------------------
question = "Show the total salary for each educational level, ordered from highest to lowest."

print("New Context Response:\n")
print(return_CCRMSQL(question, context))


New Context Response:

This is your SQL:
```sql
SELECT s.educational_level, SUM(sa.salary) AS total_salary
FROM studies s
JOIN salary sa ON s.ID_usr = sa.ID_usr
GROUP BY s.educational_level
ORDER BY total_salary DESC;
```

Explanation: This query joins the 'studies' and 'salary' tables based on the user ID, calculates the total salary for each educational level by summing up the salaries, and then orders the results from highest to lowest total salary.
